# IADD YOLOv11 - run8: AdamW + cosine LR

## 1. GPU / install / mount

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')


## 2. Config - all settings for this run

In [ ]:
# EXPERIMENT vs run7: optimizer SGD(auto)->AdamW, lr0=0.001, cosine LR decay.
# VRAM-neutral. Different optimization path + smoother end-of-training decay.
MODEL    = 'yolo11s.pt'
IMGSZ    = 1280
BATCH    = 12
EPOCHS   = 150
PATIENCE = 10
RUN_NAME = 'run8'

# all training settings live here (recorded in args.yaml)
OPTIM    = 'AdamW'
LR0      = 0.001
COS_LR   = True
MIXUP    = 0.1
HSV_V    = 0.4
DEGREES  = 5.0

DRIVE    = '/content/drive/MyDrive'
RUNS     = f'{DRIVE}/iadd_runs'
DS       = '/content/iadd_subset_v5'
DATA     = f'{DS}/data.yaml'

# resolve iadd_runs to its REAL path (Drive 'Add shortcut' lives under
# .shortcut-targets-by-id, where glob/listing works; the MyDrive shortcut does not).
import glob as _g, os as _o
RUNS = next((c for c in _g.glob('/content/drive/.shortcut-targets-by-id/*/iadd_runs') + [RUNS]
             if _o.path.isdir(c)), RUNS)
print('RUNS ->', RUNS)


## 3. Unzip the dataset (re-run after any reconnect)

In [ ]:
!rm -rf /content/iadd_subset_v5
!unzip -q "/content/drive/MyDrive/iadd_subset_v5.zip" -d /content


## 4. Train - fresh, or auto-resume from Drive

In [ ]:
from ultralytics import YOLO
import os

CKPT = f'{RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(CKPT):
    print('checkpoint found -> resuming:', CKPT)
    model = YOLO(CKPT); model.train(resume=True)
else:
    print('fresh run:', MODEL, '| imgsz', IMGSZ, '| batch', BATCH, '| opt', OPTIM)
    model = YOLO(MODEL)
    model.train(
        data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=RUN_NAME, exist_ok=True, plots=False,  # Drive-shortcut glob crashes plot_results; safeguard/eval draw plots instead
        optimizer=OPTIM, lr0=LR0, cos_lr=COS_LR, mixup=MIXUP, hsv_v=HSV_V, degrees=DEGREES,
    )


## 5. Convergence check - run any time to decide when to stop

In [ ]:
import pandas as pd
df = pd.read_csv(f'{RUNS}/{RUN_NAME}/results.csv'); df.columns = [c.strip() for c in df.columns]
fit = 0.1 * df['metrics/mAP50(B)'] + 0.9 * df['metrics/mAP50-95(B)']
best_i = int(fit.idxmax()); since = len(fit) - 1 - best_i
print(f'epochs done: {len(df)}   best at epoch {int(df["epoch"].iloc[best_i])}   ({since} ago)')
print('mAP50 last 6:', [round(x, 4) for x in df['metrics/mAP50(B)'].tail(6)])
print('\n>>> CONVERGED - safe to STOP <<<' if since >= 8 else '>>> still improving <<<')


## 6. Finish fine-tune - 10 low-LR epochs (kept only if it beats the main model)

In [ ]:
from ultralytics import YOLO
import os
# finish fine-tune: 10 low-LR epochs, mosaic+mixup OFF. Small but real gain (~+0.5 mAP50).
FT_CKPT = f'{RUNS}/{RUN_NAME}_ft/weights/last.pt'
if os.path.exists(FT_CKPT):
    print('resuming finish ->', FT_CKPT); YOLO(FT_CKPT).train(resume=True)
else:
    YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt').train(
        data=DATA, epochs=10, imgsz=IMGSZ, batch=BATCH,
        project=RUNS, name=f'{RUN_NAME}_ft', exist_ok=True, plots=False,
        optimizer='SGD', lr0=0.001, lrf=0.01, warmup_epochs=0.0,
        mosaic=0.0, mixup=0.0, hsv_v=HSV_V, degrees=DEGREES)


## 7. Evaluation - main vs ft, keep the better; honest mAP + confusion matrix

In [ ]:
from ultralytics import YOLO
import numpy as np, os, shutil
TAG = f'{RUN_NAME}_eval'

# evaluate every candidate weight honestly on val + test, then keep the better one.
# finish (ft) is included only if it exists; the winner is chosen by val mAP50 (data decides).
cands = [('main', f'{RUNS}/{RUN_NAME}/weights/best.pt')]
ft = f'{RUNS}/{RUN_NAME}_ft/weights/best.pt'
if os.path.exists(ft): cands.append(('ft', ft))

res = {}
for name, w in cands:
    m = YOLO(w)
    v = m.val(data=DATA, split='val',  plots=True, project=RUNS, name=f'{TAG}_{name}_val',  exist_ok=True)
    t = m.val(data=DATA, split='test', plots=True, project=RUNS, name=f'{TAG}_{name}_test', exist_ok=True)
    res[name] = (m, v, t)
    print('%-4s VAL mAP50=%.4f mAP50-95=%.4f | TEST mAP50=%.4f mAP50-95=%.4f'
          % (name, v.box.map50, v.box.map, t.box.map50, t.box.map))

win = max(res, key=lambda k: res[k][1].box.map50)          # higher val mAP50 wins
best, mv, mt = res[win]
print('>>> official model for %s: %s (val mAP50=%.4f, test mAP50=%.4f)'
      % (RUN_NAME, win, mv.box.map50, mt.box.map50))

# confusion matrix at F1-optimal confidence (from val), winner only
CONF = 0.25
for x, y, _, yl in mv.curves_results:
    if yl == 'F1':
        px = np.asarray(x); y = np.asarray(y); f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('operating confidence (val F1) =', CONF)
for split in ('val', 'test'):
    best.val(data=DATA, split=split, conf=CONF, plots=True, project=RUNS, name=f'{TAG}_{split}_tmp', exist_ok=True)
    for f in ('confusion_matrix.png', 'confusion_matrix_normalized.png'):
        shutil.copy(f'{RUNS}/{TAG}_{split}_tmp/{f}', f'{RUNS}/{TAG}_{win}_{split}/{f}')
    shutil.rmtree(f'{RUNS}/{TAG}_{split}_tmp')
print('done. winner=%s  matrix at conf %.3f' % (win, CONF))


## 8. Session safeguard - run at the END of every session

In [ ]:
# session safeguard - RUN AT THE END OF EVERY SESSION.
# Rebuilds the full training curve from BOTH the results*.csv fragments AND the
# history stored inside the checkpoints (train_results), so a Drive-deleted
# fragment is always recoverable. Also snapshots best.pt under a unique name.
import glob, os, shutil, time
import pandas as pd, torch
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def find_run_dir(name):                       # Drive "Add shortcut" lands under .shortcut-targets-by-id
    pats = [f'/content/drive/MyDrive/iadd_runs/{name}',
            f'/content/drive/.shortcut-targets-by-id/*/iadd_runs/{name}',
            f'/content/drive/*/iadd_runs/{name}']
    cands = [c for p in pats for c in glob.glob(p)]
    good = [c for c in cands if glob.glob(f'{c}/results*.csv') or os.path.exists(f'{c}/weights/last.pt')]
    return (good or cands or [f'{RUNS}/{name}'])[0]

RUN = find_run_dir(RUN_NAME)
print('using run dir:', RUN)

if os.path.exists(f'{RUN}/weights/best.pt'):
    dst = f'{RUN}/weights/best_{time.strftime("%m%d_%H%M")}.pt'
    shutil.copy(f'{RUN}/weights/best.pt', dst); print('snapshot ->', dst)

parts = []
for f in sorted(glob.glob(f'{RUN}/results*.csv')):
    d = pd.read_csv(f); d.columns = [c.strip() for c in d.columns]; parts.append(d)
for w in ('last.pt', 'best.pt'):
    p = f'{RUN}/weights/{w}'
    if os.path.exists(p):
        tr = torch.load(p, map_location='cpu', weights_only=False).get('train_results')
        if tr:
            d = pd.DataFrame(tr); d.columns = [c.strip() for c in d.columns]; parts.append(d)

if not parts:
    raise SystemExit(f'no results under {RUN} | contents: {os.listdir(RUN) if os.path.isdir(RUN) else "MISSING"}')

m = pd.concat(parts, ignore_index=True).drop_duplicates('epoch', keep='last').sort_values('epoch')
m['epoch'] = m['epoch'].astype(int)
m.to_csv(f'{RUN}/results.csv', index=False); m.to_csv(f'{RUN}/results_master.csv', index=False)
ep = m['epoch'].tolist(); gaps = [e for e in range(ep[0], ep[-1]+1) if e not in ep]
print(f'complete curve: epochs {ep[0]}-{ep[-1]} | missing: {gaps or "none"}')

panels = ['train/box_loss','train/cls_loss','train/dfl_loss','metrics/precision(B)','metrics/recall(B)',
          'val/box_loss','val/cls_loss','val/dfl_loss','metrics/mAP50(B)','metrics/mAP50-95(B)']
fig, axs = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axs.ravel(), panels):
    s = m[col]
    ax.plot(m['epoch'], s.values, '-', color='#1f4e79', lw=1.6, marker='.', ms=4)
    ax.plot(m['epoch'], s.rolling(5, center=True, min_periods=1).mean().values, ':', color='#e08a00', lw=1.3)
    ax.set_title(col, fontsize=11); ax.grid(alpha=0.25); ax.set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{RUN}/results.png', dpi=140); plt.close()
print('results.png rebuilt (clean, no legend overlap)')


## 9. Demo - 20 best predictions from 20 different test videos

In [ ]:
# demo - 20 of the model's BEST predictions, each from a DIFFERENT test video.
# picks clean, confident detections (nice for slides), avoids empty / overcrowded frames.
import glob, os, random
from collections import defaultdict
from ultralytics import YOLO
from PIL import Image

random.seed(0)
model = YOLO(f'{RUNS}/{RUN_NAME}/weights/best.pt')
OUT = f'{RUNS}/{RUN_NAME}_demo'; os.makedirs(OUT, exist_ok=True)

by_vid = defaultdict(list)
for p in glob.glob(f'{DS}/images/test/*.jpg'):
    by_vid[os.path.basename(p).split('__')[0]].append(p)

cand = []
for vid, ps in by_vid.items():
    best_s, best_p, best_r = -1, None, None
    for p in random.sample(ps, min(4, len(ps))):
        r = model.predict(p, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
        cf = r.boxes.conf.cpu().numpy() if r.boxes is not None else []
        if len(cf) == 0: continue
        hi = int((cf >= 0.6).sum())
        score = float(cf.mean()) * min(hi, 8)          # confident + several clean boxes
        if 2 <= hi <= 15 and score > best_s:
            best_s, best_p, best_r = score, p, r
    if best_p is not None: cand.append((best_s, os.path.basename(best_p), best_r))

cand.sort(key=lambda t: t[0], reverse=True)
picks = cand[:20]
print(f'{len(by_vid)} test videos -> {len(picks)} best demo images')

saved = []
for s, name, r in picks:
    out = f'{OUT}/{name}'; r.save(filename=out); saved.append(out)

cols, rows, w, h = 5, 4, 480, 270
sheet = Image.new('RGB', (w*cols, h*rows), 'white')
for k, sp in enumerate(saved):
    sheet.paste(Image.open(sp).convert('RGB').resize((w, h)), ((k % cols)*w, (k // cols)*h))
sheet.save(f'{OUT}/_contact_sheet.jpg', quality=92)
print('saved best demo images + _contact_sheet.jpg ->', OUT)
